# Stage 4 — Seed-42 patience-4 pilot

Validation-only adjustment run. Reuses the exact approved full-corpus mined-negative artifact and verified batch-32 smoke result. The sole experimental change is early-stopping patience from two to four evaluations; evaluation/save frequency remains every five steps and checkpoint selection remains validation nDCG@10. Test files are never loaded. No three-seed run is performed.


In [ ]:
%pip install -q "sentence-transformers==6.0.1" "transformers==5.16.1" "faiss-cpu==1.15.0" "rank-bm25==0.2.2" datasets pandas


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import base64
import hashlib
import io
import zipfile
from pathlib import Path

PAYLOAD_SHA256 = '25ADAA3CDCF9130C016704EA116EA34A39193E71CCA46C3BF224286D3741AD81'
PAYLOAD_B64 = '...'
payload = base64.b64decode(PAYLOAD_B64)
assert hashlib.sha256(payload).hexdigest().upper() == PAYLOAD_SHA256
SOURCE_DIR = Path('/content/drive/MyDrive/RAFT/stage4_hardneg/patience4_source_snapshot')
SOURCE_DIR.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(payload)) as archive:
    archive.extractall(SOURCE_DIR)
print(f'STAGE4_PATIENCE4_SOURCE_READY={SOURCE_DIR}')
print(f'PAYLOAD_SHA256={PAYLOAD_SHA256}')


In [ ]:
import copy
import json
import subprocess
import sys
import torch

DRIVE_ROOT = Path('/content/drive/MyDrive/RAFT')
ORIGINAL_CONFIG = json.loads((SOURCE_DIR / 'configs/stage4_hardneg_pilot.json').read_text())
CONFIG_PATH = SOURCE_DIR / 'configs/stage4_hardneg_pilot_patience4.json'
CONFIG = json.loads(CONFIG_PATH.read_text())
required = [
    DRIVE_ROOT / 'stage1_scifact/processed/corpus.jsonl',
    DRIVE_ROOT / 'stage1_scifact/processed/queries_train.jsonl',
    DRIVE_ROOT / 'stage1_scifact/processed/qrels_train.tsv',
    DRIVE_ROOT / 'stage1_scifact/processed/queries_validation.jsonl',
    DRIVE_ROOT / 'stage1_scifact/processed/qrels_validation.tsv',
    DRIVE_ROOT / 'stage2_baselines/results/stage2_baselines.csv',
    DRIVE_ROOT / 'stage3_inbatch/final_results/stage3_final_summary.json',
    DRIVE_ROOT / 'stage4_hardneg/mined/stage4_seed42_hard_negatives.jsonl',
    DRIVE_ROOT / 'stage4_hardneg/mined/stage4_seed42_mining_summary.json',
    DRIVE_ROOT / 'stage4_hardneg/smoke_seed42/results/stage4_smoke_result.json',
]
missing = [str(path) for path in required if not path.exists()]
assert not missing, f'Missing required Drive artifacts: {missing}'
assert torch.cuda.is_available(), 'Select a Colab GPU runtime before running the pilot.'
assert ORIGINAL_CONFIG['model'] == CONFIG['model']
assert ORIGINAL_CONFIG['dataset'] == CONFIG['dataset']
assert ORIGINAL_CONFIG['mining'] == CONFIG['mining']
assert ORIGINAL_CONFIG['smoke_test'] == CONFIG['smoke_test']
assert ORIGINAL_CONFIG['evaluation'] == CONFIG['evaluation']
old_training = copy.deepcopy(ORIGINAL_CONFIG['training'])
new_training = copy.deepcopy(CONFIG['training'])
assert old_training['early_stopping_rule'].pop('patience_evaluations') == 2
assert new_training['early_stopping_rule'].pop('patience_evaluations') == 4
assert old_training == new_training
print(f'GPU={torch.cuda.get_device_name(0)}')
print('ONLY_EXPERIMENTAL_CHANGE: EARLY-STOPPING PATIENCE 2 -> 4')


In [ ]:
subprocess.run([
    sys.executable,
    str(SOURCE_DIR / 'src/run_stage4_patience4_pilot.py'),
    '--repo-dir', str(SOURCE_DIR),
    '--drive-root', str(DRIVE_ROOT),
], check=True)


In [ ]:
import pandas as pd

RESULT_DIR = DRIVE_ROOT / 'stage4_hardneg/pilot_seed42_patience4/results'
PILOT_RESULT = RESULT_DIR / 'stage4_pilot_result.json'
COMPARISON = RESULT_DIR / 'stage4_pilot_validation_comparison.csv'
pilot = json.loads(PILOT_RESULT.read_text())
assert pilot['test_split_loaded'] is False
assert pilot['fresh_pretrained_initialization'] is True
assert pilot['config']['training']['early_stopping_rule']['patience_evaluations'] == 4
trajectory = []
for row in pilot['validation_evaluations']:
    trajectory.append({
        'step': row['step'],
        'epoch': row['epoch'],
        'recall_at_5': row['eval_scifact_validation_cosine_recall@5'],
        'recall_at_10': row['eval_scifact_validation_cosine_recall@10'],
        'precision_at_5': row['eval_scifact_validation_cosine_precision@5'],
        'precision_at_10': row['eval_scifact_validation_cosine_precision@10'],
        'mrr_at_10': row['eval_scifact_validation_cosine_mrr@10'],
        'ndcg_at_10': row['eval_scifact_validation_cosine_ndcg@10'],
    })
print('FULL CHECKPOINT-BY-CHECKPOINT VALIDATION TRAJECTORY:')
display(pd.DataFrame(trajectory))
print('\nFINAL BEST-CHECKPOINT COMPARISON:')
display(pd.read_csv(COMPARISON))
print('\nLOSS SUMMARY:')
print(json.dumps(pilot['loss_summary'], indent=2, sort_keys=True))
print(f"\nBEST CHECKPOINT SOURCE: {pilot['best_checkpoint_source']}")
print(f"SAVED BEST CHECKPOINT: {pilot['saved_best_checkpoint']}")
print(f"GLOBAL STEPS: {pilot['global_steps_completed']}")
print(f"EPOCHS COMPLETED: {pilot['epochs_completed']}")
print(f"TEST SPLIT LOADED: {pilot['test_split_loaded']}")
print(f'Attach or paste this result in the Codex task: {PILOT_RESULT}')
